In [33]:
# 데이터 생성
import pandas as pd
dataset = pd.read_csv('../../data/csv/movie_reviews2.csv')
dataset.head()

,label,review
0,1,영화가 너무 재미있어서 하나도 안 지루해요.
1,0,영화가 하나도 안 재미있어서 너무 지루해요.
2,1,기대 안 했는데 생각보다 너무 괜찮네요.
3,0,기대 많이 했는데 생각보다 너무 별로네요.
4,1,배우 연기는 별로지만 스토리가 살렸습니다.


In [34]:
# 독립 종속(답) 분리

X = dataset['review'].to_numpy()
y = dataset['label'].to_numpy().astype('float32')


In [35]:
# 텍스트 전처리
# 형태소 분석

from konlpy.tag import Okt
okt = Okt()

# ---------------------------------------------------------------
# 문장을 형태소 별로 분리해서 공백으로 구분하여 다시 한문장으로 만드는 함수
# ---------------------------------------------------------------


def tokenize_korean(text_list):
 return [" ".join(okt.morphs(text, stem=True)) for text in text_list]
# print(tokenize_korean(["오늘은 학교에 갔습니다."]))
korean_morphs = tokenize_korean(X)

In [36]:
# 벡터화
from tensorflow.keras import layers, models
vectorize_layer = layers.TextVectorization(
  max_tokens=1000,
  output_mode = 'int',
  output_sequence_length=10
)

# 데이터를 학습 시켜 단어 사전을 만듬
vectorize_layer.adapt(korean_morphs)

In [37]:
import tensorflow as tf

# from_tensor_slices : korean_morphs와 y에서  하나씩 꺼내 하나의 세트로 묶음
train_ds = tf.data.Dataset.from_tensor_slices((korean_morphs, y)).batch(8)

In [38]:
def bulid_transform_model():
  # 입력층
  inputs = layers.Input((1,), dtype=tf.string)
  
	# 벡터화와 임베딩
  x = vectorize_layer(inputs)
  vocab_size = vectorize_layer.vocabulary_size()
  x = layers.Embedding(input_dim=vocab_size, output_dim=64)(x)
  
	# 멀티헤드어텐션
  attention_output = layers.MultiHeadAttention(num_heads=2, key_dim=64)(x,x)
  
	# 잔차 연결 및 정규화
  # 잔차 연결 : 어텐션 결과에 원래 입력값을 더해 정보를 보존 
  x = layers.Add()([x, attention_output])
  x = layers.LayerNormalization()(x)
  
	# 추가학습
  ffn = layers.Dense(64, activation='relu')(x)
  
	# 잔차 연결 및 정규화
  x = layers.Add()([x, ffn])
  x = layers.LayerNormalization()(x)
  
	# 출력층
  x = layers.GlobalAveragePooling1D()(x)
  outputs = layers.Dense(1, activation='sigmoid')(x)
  return models.Model(inputs=inputs, outputs=outputs)

# 모델 설계
model = bulid_transform_model()


In [39]:
# 모델 설정
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [40]:
model.fit(train_ds, epochs=50)

Epoch 1/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.3667 - loss: 0.7800
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7333 - loss: 0.5970 
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7667 - loss: 0.4901 
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8333 - loss: 0.4114 
Epoch 5/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9000 - loss: 0.3360 
Epoch 6/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9333 - loss: 0.2708
Epoch 7/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9333 - loss: 0.2167 
Epoch 8/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9333 - loss: 0.1704 
Epoch 9/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9333 - loss: 0.1388
Epoch 10/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9333 - loss: 0.1209
Epoch 11/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9333 - loss: 0.1118 
Epoch 12/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9333 - loss: 0.1202 
Epoch

In [41]:
test_text = ["영화가 너무 재미있어서 하나도 안 지루해요"]

# 형태소별로 문장을 수정
test_morphs = tokenize_korean(test_text)
test_morphs

# 테스트 할 때 텐서 타입을 변환(문자열 이슈)
test_input = tf.constant(test_morphs)

# 예측 실행
predictions = model.predict(test_input)
predictions_size = len(predictions)
for i in range(predictions_size):
  p = predictions[i][0] * 100
  print(f'{test_text[i]} : {'긍정' if p > 50 else '부정'}({p:.2f}%)')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step
영화가 너무 재미있어서 하나도 안 지루해요 : 긍정(51.80%)
